# Config Recipes and Operations

This notebook collects reusable snippets from the old scratch `run_model.ipynb`.

Use it as a reference for changing strategy config, signal bars, execution instruments, double-down rules, sizing, and order/state inspection. The runnable live examples are still the focused notebooks:

- `catboost_1min_stock.ipynb`
- `catboost_1min_calls.ipynb`
- `mean_reversion_dollar_bars.ipynb`

## Imports and Base Config

In [ ]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == "examples":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import DEFAULT_LIVE_CONFIG, LiveTradingConfig
from execution import active_orders, all_orders, filled_orders, recent_fills, strategy_open_trades

raw = copy.deepcopy(DEFAULT_LIVE_CONFIG)
config = LiveTradingConfig.from_dict(raw)
config.symbol


## CatBoost On / Off

In [ ]:
raw["model"]["enabled"] = True
raw["model"]["prob_threshold"] = 0.55
raw["paths"]["catboost_model_path"] = "models/live/catboost/meta_catboost.cbm"

config = LiveTradingConfig.from_dict(raw)
config.model


In [ ]:
raw["model"]["enabled"] = False
config = LiveTradingConfig.from_dict(raw)
config.model


## Signal Bar Recipes

Feature windows are counted in completed signal bars. With `5min` bars, `window=20` means 20 five-minute bars. With dollar bars, `window=20` means 20 completed dollar bars.

In [ ]:
# 1-minute time bars
raw["features"]["bar"] = {
    "type": "time",
    "timeframe": "1min",
    "history_window": 390,
}

config = LiveTradingConfig.from_dict(raw)
config.features["bar"]


In [ ]:
# 5-minute time bars
raw["features"]["bar"] = {
    "type": "time",
    "timeframe": "5min",
    "history_window": 78,
}

config = LiveTradingConfig.from_dict(raw)
config.features["bar"]


In [ ]:
# Dollar bars
raw["features"]["bar"] = {
    "type": "dollar",
    "dollar_threshold": 1_000_000,
    "history_window": 390,
}

config = LiveTradingConfig.from_dict(raw)
config.features["bar"]


## Entry Strategy Recipes

In [ ]:
# Original median/MAD entry
raw["strategy"]["entry_strategy"] = {
    "name": "median_mad",
}
raw["strategy"]["entry_conditions"] = {}

config = LiveTradingConfig.from_dict(raw)
config.strategy["entry_strategy"]


In [ ]:
# Standard Bollinger Bands entry, not median-based
raw["strategy"]["entry_strategy"] = {
    "name": "standard_bb",
    "z": 2.0,
    "double_down_mult": 3.0,
}
raw["strategy"]["entry_conditions"] = {}

config = LiveTradingConfig.from_dict(raw)
config.strategy["entry_strategy"]


In [ ]:
# Always-true entry predicate.
# This turns the signal layer into "enter whenever risk, timing, model,
# and hard price limits allow it". Double-down is controlled by
# double_down.price_rules, which makes it useful for buy-the-dip logic.
raw["strategy"]["entry_strategy"] = {
    "name": "always_true",
}
raw["strategy"]["entry_conditions"] = {}
raw["strategy"]["entry_price_limits"] = {
    "first_entry": {
        "max_stock_price": 500.0,
        "max_buy_price": 500.0,
    },
}
raw["double_down"]["price_rules"] = [
    {"basis": "underlying", "mode": "abs", "max_change": -5.00},
    {"basis": "execution", "mode": "abs", "max_change": -5.00},
]

config = LiveTradingConfig.from_dict(raw)
config.strategy["entry_strategy"], config.strategy["entry_price_limits"], config.double_down["price_rules"]


In [ ]:
# Fully custom raw feature rules. Every rule in the selected list must pass.
raw["strategy"]["entry_conditions"] = {
    "first_entry": [
        {"field": "standard_bb_z", "op": "<=", "value": -2.0},
        {"field": "rsi_14", "op": "<=", "value": 35.0},
    ],
    "double_down": [
        {"field": "standard_bb_z", "op": "<=", "value": -6.0},
    ],
}

config = LiveTradingConfig.from_dict(raw)
config.strategy["entry_conditions"]


## Execution Instrument Recipes

Signals are still based on the underlying stock. These settings only change what the strategy trades.

In [ ]:
# Trade stock
raw["execution"]["instrument"] = {
    "type": "stock",
    "exchange": "SMART",
    "currency": "USD",
    "limit_entry_offset_pct": 0.0005,
    "limit_exit_offset_pct": 0.0005,
}

config = LiveTradingConfig.from_dict(raw)
config.execution["instrument"]


In [ ]:
# Trade one call option
raw["execution"]["instrument"] = {
    "type": "option",
    "exchange": "SMART",
    "currency": "USD",
    "expiry": "20260619",
    "strike": 500.0,
    "right": "C",
    "multiplier": 100.0,
    "limit_entry_offset_pct": 0.02,
    "limit_exit_offset_pct": 0.02,
}

config = LiveTradingConfig.from_dict(raw)
config.execution["instrument"]


## Double-Down Recipes

In [ ]:
# Require underlying stock to be down at least 1% from previous entry.
raw["double_down"]["enabled"] = True
raw["double_down"]["price_rules"] = [
    {"basis": "underlying", "mode": "pct", "max_change": -0.01},
]

config = LiveTradingConfig.from_dict(raw)
config.double_down


In [ ]:
# For options: require underlying and execution instrument movement.
raw["double_down"]["enabled"] = True
raw["double_down"]["price_rules"] = [
    {"basis": "underlying", "mode": "pct", "max_change": -0.01},
    {"basis": "execution", "mode": "pct", "max_change": -0.20},
    # {"basis": "execution", "mode": "abs", "max_change": -0.50},
]

config = LiveTradingConfig.from_dict(raw)
config.double_down


In [ ]:
# Disable double-down entirely.
raw["double_down"]["enabled"] = False
config = LiveTradingConfig.from_dict(raw)
config.double_down


## Latest Buy/Sell Dip Guard

This guard compares first entries against whichever happened most recently: last buy or last sell. Double-down entries compare against the last buy. It can compare the underlying stock price, the execution instrument price, or both.

In [ ]:
# Require first entries to be lower than the most recent buy/sell event.
# Require double-down entries to be at least $5 below the last buy.
raw["strategy"]["latest_trade_price_rules"] = {
    "enabled": True,
    "first_entry": [
        {"basis": "underlying", "mode": "pct", "max_change": -0.01},
    ],
    "double_down": [
        {"basis": "execution", "mode": "abs", "max_change": -5.00},
    ],
}

config = LiveTradingConfig.from_dict(raw)
config.strategy["latest_trade_price_rules"]


In [ ]:
# For options, require the first-entry contract buy price to be lower than
# the most recent buy/sell execution price, and double-down buys to be at
# least $0.50 below the last buy execution price.
raw["strategy"]["latest_trade_price_rules"] = {
    "enabled": True,
    "first_entry": [
        {"basis": "underlying", "mode": "pct", "max_change": -0.01},
        {"basis": "execution", "mode": "pct", "max_change": -0.20},
    ],
    "double_down": [
        {"basis": "execution", "mode": "abs", "max_change": -0.50},
    ],
}

config = LiveTradingConfig.from_dict(raw)
config.strategy["latest_trade_price_rules"]


## Market Session Recipes

`useRTH` controls the IB real-time bar subscription. `outside_rth` controls whether orders may be routed outside regular trading hours. `exchange` can be `SMART` for regular/extended routing or `OVERNIGHT` for overnight stock routing where supported by IBKR.

In [ ]:
# Regular-hours stock bars and regular-hours order intent.
raw["execution"]["market_data"] = {
    "exchange": "SMART",
    "barSize": 5,
    "whatToShow": "TRADES",
    "useRTH": True,
}
raw["execution"]["outside_rth"] = False
raw["execution"]["instrument"]["exchange"] = "SMART"

config = LiveTradingConfig.from_dict(raw)
config.execution["market_data"], config.execution["outside_rth"], config.execution["instrument"]


In [ ]:
# Premarket/after-hours stock bars and outside-RTH order permission.
raw["execution"]["market_data"] = {
    "exchange": "SMART",
    "barSize": 5,
    "whatToShow": "TRADES",
    "useRTH": False,
}
raw["execution"]["outside_rth"] = True
raw["execution"]["instrument"]["exchange"] = "SMART"

config = LiveTradingConfig.from_dict(raw)
config.execution["market_data"], config.execution["outside_rth"], config.execution["instrument"]


In [ ]:
# Overnight stock bars/orders where IBKR supports OVERNIGHT for the symbol.
raw["execution"]["market_data"] = {
    "exchange": "OVERNIGHT",
    "barSize": 5,
    "whatToShow": "TRADES",
    "useRTH": False,
}
raw["execution"]["outside_rth"] = True
raw["execution"]["instrument"]["exchange"] = "OVERNIGHT"

config = LiveTradingConfig.from_dict(raw)
config.execution["market_data"], config.execution["outside_rth"], config.execution["instrument"]


## Risk, Sizing, Timing, and Saving

In [ ]:
raw["sizing"]["default_qty"] = 100
raw["sizing"]["size_schedule"] = {
    0: 100,
    1: 100,
    2: 200,
    3: 300,
}

config = LiveTradingConfig.from_dict(raw)
config.sizing


In [ ]:
raw["strategy"]["max_open_trades_today"] = 4
raw["strategy"]["max_new_trades_per_day"] = 4
raw["strategy"]["max_open_trades_total"] = 200
raw["strategy"]["no_trade_first_minutes"] = 60
raw["strategy"]["no_new_entries_last_minutes"] = 60
raw["strategy"]["min_take_profit"] = 0.01
raw["strategy"]["reentry_cooldown_days"] = 7
raw["strategy"]["reentry_discount_pct"] = 0.01

config = LiveTradingConfig.from_dict(raw)
config.strategy


In [ ]:
raw["paths"]["open_trades_path"] = "META_open_trades.csv"
raw["saving"]["autosave_open_trades"] = True

config = LiveTradingConfig.from_dict(raw)
config.paths, config.saving


## Manual Operations

These cells assume you already created `ib` and `algo` in one of the live example notebooks.

In [ ]:
# Strategy state saved on disk.
# algo.open_trades_store.save(algo.open_trades)
# algo.open_trades = algo.open_trades_store.load()
# strategy_open_trades(algo)


In [ ]:
# IB order views.
# active_orders(ib)
# filled_orders(ib)
# recent_fills(ib)
# all_orders(ib)


In [ ]:
# Avoid duplicate callbacks after rerunning setup cells.
# real_time_bars.updateEvent -= algo.on_bar
# real_time_bars.updateEvent += algo.on_bar
